# Assignment 3 Main Project: Problem-based Group Project

## Topic: Use Case 1 — Intelligent Customer Service Systems

### Part: RAG + Agent Engineer (Person 4)

**Name:** Ayan Roy

**Student ID:** 49128299

**Role:** P4 — RAG + Agent Engineer

This notebook implements **Part 4** of the group project: **RAG + Agent Engineer**.

The purpose of this part is to give the customer service chatbot access to company-specific knowledge and simple decision-making ability.

Part 4 connects with:

- **Part 2:** Basic NLP analysis, including category, sentiment, and entities.
- **Part 3:** LLM prompt engineering and customer response generation.
- **Part 5:** Final Streamlit or Gradio interface and full system integration.

The main components implemented in this notebook are:

1. Knowledge base creation
2. Embedding generation
3. FAISS vector database
4. RAG retrieval pipeline
5. Escalation classifier
6. Agent decision logic
7. Dialogue tracking
8. Final integration function for the full system


## 1. Install and Import Libraries

This notebook uses:

- **FAISS** for vector search / vector database
- **SentenceTransformer** for semantic embeddings
- **Hugging Face datasets** to load the Bitext customer support dataset
- **Pandas/Numpy** for data handling

Run the install cell below only if these packages are missing.


In [1]:
# Run this cell only if needed.
# If everything already imports correctly, you can skip it.

!pip install faiss-cpu sentence-transformers datasets pandas numpy scikit-learn



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [3]:
import re
import json
import numpy as np
import pandas as pd
from datetime import datetime

# FAISS vector database
try:
    import faiss
    FAISS_AVAILABLE = True
    print("FAISS imported successfully.")
except ImportError:
    FAISS_AVAILABLE = False
    print("FAISS is not installed. Run: pip install faiss-cpu")

# SentenceTransformer embeddings
try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
    print("SentenceTransformer imported successfully.")
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    print("sentence-transformers is not installed. Run: pip install sentence-transformers")

# Hugging Face dataset loader
try:
    from datasets import load_dataset
    DATASETS_AVAILABLE = True
    print("datasets imported successfully.")
except ImportError:
    DATASETS_AVAILABLE = False
    print("datasets is not installed. Run: pip install datasets")

# Fallback embedding method if sentence-transformers is unavailable
from sklearn.feature_extraction.text import TfidfVectorizer


FAISS imported successfully.


c:\Users\royay\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


SentenceTransformer imported successfully.
datasets imported successfully.


## 2. Load the Customer Support Dataset

This notebook uses the Bitext customer support dataset.

The dataset columns used for the knowledge base are:

- `instruction`: customer inquiry
- `response`: support answer
- `category`: broad support category
- `intent`: specific customer intent

These columns are used directly because they match the dataset structure used in Part 2. The dataset is used to create realistic FAQ-style knowledge base entries for retrieval.


In [4]:
# Load the same Bitext customer support dataset used in Part 2

df = pd.DataFrame(
    load_dataset(
        "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
        split="train[:20%]"
    )
)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()


Dataset loaded successfully.
Dataset shape: (5374, 5)
Columns: ['flags', 'instruction', 'category', 'intent', 'response']


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


## 3. Build Dataset-Based FAQ Knowledge Base

Each dataset row is converted into a FAQ-style RAG entry.

This creates a better knowledge base than only using manually written policies because the entries are based on customer-service-style data.



In [5]:
def build_dataset_faq_entries(df, examples_per_intent=8, random_state=42):
    """
    Convert customer support dataset rows into FAQ-style RAG entries.

    The Bitext dataset columns used here are:
    - instruction
    - response
    - category
    - intent
    """

    # Use the known dataset columns directly
    kb_source = df[["instruction", "response", "category", "intent"]].dropna().copy()

    # Convert all fields to string for safe formatting
    for col in ["instruction", "response", "category", "intent"]:
        kb_source[col] = kb_source[col].astype(str)

    # Remove empty rows
    kb_source = kb_source[
        (kb_source["instruction"].str.strip() != "") &
        (kb_source["response"].str.strip() != "")
    ].copy()

    # Shuffle first, then take up to N examples per intent
    kb_sample = (
        kb_source
        .sample(frac=1, random_state=random_state)
        .groupby("intent", as_index=False, group_keys=False)
        .head(examples_per_intent)
        .reset_index(drop=True)
    )

    entries = []

    for i, row in kb_sample.iterrows():
        entry = {
            "doc_id": f"dataset_faq_{i:04d}",
            "source_type": "dataset_faq",
            "title": f"{row['category']} - {row['intent']}",
            "category": row["category"],
            "intent": row["intent"],
            "question": row["instruction"],
            "answer": row["response"],
            "text": (
                f"Customer question: {row['instruction']}\n"
                f"Recommended support answer: {row['response']}\n"
                f"Category: {row['category']}\n"
                f"Intent: {row['intent']}"
            )
        }
        entries.append(entry)

    return entries


dataset_faq_entries = build_dataset_faq_entries(df, examples_per_intent=8)

print("Dataset-based FAQ entries:", len(dataset_faq_entries))
pd.DataFrame(dataset_faq_entries).head()


Dataset-based FAQ entries: 48


,doc_id,source_type,title,category,intent,question,answer,text
0,dataset_faq_0000,dataset_faq,ORDER - cancel_order,ORDER,cancel_order,"I can't pay for this item, cancel order {{Orde...",I understand the urgency and frustration of no...,"Customer question: I can't pay for this item, ..."
1,dataset_faq_0001,dataset_faq,PAYMENT - check_payment_methods,PAYMENT,check_payment_methods,"I have got to list the payment modalities, hel...",Thank you for reaching out! I completely under...,Customer question: I have got to list the paym...
2,dataset_faq_0002,dataset_faq,SHIPPING - change_shipping_address,SHIPPING,change_shipping_address,can you tell me more about updating the address?,Of course! When it comes to updating your addr...,Customer question: can you tell me more about ...
3,dataset_faq_0003,dataset_faq,INVOICE - check_invoice,INVOICE,check_invoice,wanna see the bill from {{Person Name}},I'll get right on it! I understand your desire...,Customer question: wanna see the bill from {{P...
4,dataset_faq_0004,dataset_faq,INVOICE - check_invoice,INVOICE,check_invoice,ser the bill #85632,I'll do my best! I'm here to assist you with r...,Customer question: ser the bill #85632\nRecomm...


## 4. Add Manual Company Policy Entries

The final knowledge base is **hybrid**:

```text
Dataset-based FAQ entries + Manual company policy entries
```

Dataset entries help the system match realistic customer questions.

Manual policies help the system provide official support rules.


In [6]:
manual_policy_entries = [
    {
        "doc_id": "policy_refund_001",
        "source_type": "manual_policy",
        "title": "Refund Policy",
        "category": "REFUND",
        "intent": "refund_policy",
        "question": "When can a customer request a refund?",
        "answer": (
            "Customers may request a refund within 14 days of purchase if the product is damaged, "
            "not delivered, incorrect, or if the customer was charged incorrectly. Refund requests "
            "should include the order ID, product name, payment amount, and reason for refund."
        ),
        "text": (
            "Refund Policy: Customers may request a refund within 14 days of purchase if the product is damaged, "
            "not delivered, incorrect, or if the customer was charged incorrectly. Refund requests should include "
            "the order ID, product name, payment amount, and reason for refund."
        )
    },
    {
        "doc_id": "policy_delivery_001",
        "source_type": "manual_policy",
        "title": "Shipping and Delivery Policy",
        "category": "DELIVERY",
        "intent": "shipping_policy",
        "question": "What should happen when an order has not arrived?",
        "answer": (
            "Standard delivery usually takes 3 to 7 business days. If an order has not arrived after the estimated "
            "delivery date, customers should provide the order ID and delivery address. The case may be escalated "
            "to the logistics team if tracking is unavailable or the package appears lost."
        ),
        "text": (
            "Shipping and Delivery Policy: Standard delivery usually takes 3 to 7 business days. If an order has not "
            "arrived after the estimated delivery date, customers should provide the order ID and delivery address. "
            "The case may be escalated to the logistics team if tracking is unavailable or the package appears lost."
        )
    },
    {
        "doc_id": "policy_damaged_001",
        "source_type": "manual_policy",
        "title": "Damaged Product Policy",
        "category": "PRODUCT",
        "intent": "damaged_product_policy",
        "question": "What should happen if a product arrives damaged?",
        "answer": (
            "If a customer receives a damaged product, they should contact support within 7 days of delivery. "
            "They may be asked to provide photos of the damaged item, the order ID, and a short description of the damage."
        ),
        "text": (
            "Damaged Product Policy: If a customer receives a damaged product, they should contact support within 7 days "
            "of delivery. They may be asked to provide photos of the damaged item, the order ID, and a short description "
            "of the damage."
        )
    },
    {
        "doc_id": "policy_billing_001",
        "source_type": "manual_policy",
        "title": "Billing and Payment Policy",
        "category": "BILLING",
        "intent": "billing_policy",
        "question": "How should billing and duplicate charge issues be handled?",
        "answer": (
            "For duplicate charges, incorrect charges, or failed payments, customers should provide the order ID, "
            "transaction date, payment method, and charged amount. Possible overcharging should be reviewed by the billing team."
        ),
        "text": (
            "Billing and Payment Policy: For duplicate charges, incorrect charges, or failed payments, customers should "
            "provide the order ID, transaction date, payment method, and charged amount. Possible overcharging should be "
            "reviewed by the billing team."
        )
    },
    {
        "doc_id": "policy_account_001",
        "source_type": "manual_policy",
        "title": "Account and Login Support Policy",
        "category": "ACCOUNT",
        "intent": "account_login_policy",
        "question": "What should happen if a customer cannot log in?",
        "answer": (
            "If a customer cannot log in, they should first reset their password using the registered email address. "
            "If password reset does not work, support should verify the customer's identity and check whether the account "
            "is locked, suspended, or affected by a technical error."
        ),
        "text": (
            "Account and Login Support Policy: If a customer cannot log in, they should first reset their password using "
            "the registered email address. If password reset does not work, support should verify the customer's identity "
            "and check whether the account is locked, suspended, or affected by a technical error."
        )
    },
    {
        "doc_id": "policy_technical_001",
        "source_type": "manual_policy",
        "title": "Technical Support Policy",
        "category": "TECHNICAL_SUPPORT",
        "intent": "technical_support_policy",
        "question": "How should technical support issues be handled?",
        "answer": (
            "For technical issues, customers should describe the problem, device model, software version, and steps already tried. "
            "Support should suggest basic troubleshooting such as restarting the device, updating the app, clearing cache, "
            "or checking the internet connection."
        ),
        "text": (
            "Technical Support Policy: For technical issues, customers should describe the problem, device model, software version, "
            "and steps already tried. Support should suggest basic troubleshooting such as restarting the device, updating the app, "
            "clearing cache, or checking the internet connection."
        )
    },
    {
        "doc_id": "policy_escalation_001",
        "source_type": "manual_policy",
        "title": "Escalation Policy",
        "category": "ESCALATION",
        "intent": "escalation_policy",
        "question": "When should a case be escalated to a human agent?",
        "answer": (
            "Cases should be escalated when the customer is very angry, requests a manager, mentions legal action, reports repeated "
            "unresolved issues, has a high-value billing complaint, or when the system cannot retrieve a relevant policy."
        ),
        "text": (
            "Escalation Policy: Cases should be escalated when the customer is very angry, requests a manager, mentions legal action, "
            "reports repeated unresolved issues, has a high-value billing complaint, or when the system cannot retrieve a relevant policy."
        )
    },
    {
        "doc_id": "policy_privacy_001",
        "source_type": "manual_policy",
        "title": "Privacy and Data Handling Policy",
        "category": "PRIVACY",
        "intent": "privacy_policy",
        "question": "What information should not be requested in customer service chat?",
        "answer": (
            "The system should not ask for passwords, full card numbers, or unnecessary sensitive personal data. If secure verification "
            "is required, the customer should be moved to a secure human support channel."
        ),
        "text": (
            "Privacy and Data Handling Policy: The system should not ask for passwords, full card numbers, or unnecessary sensitive "
            "personal data. If secure verification is required, the customer should be moved to a secure human support channel."
        )
    }
]

print("Manual policy entries:", len(manual_policy_entries))
pd.DataFrame(manual_policy_entries).head()


Manual policy entries: 8


,doc_id,source_type,title,category,intent,question,answer,text
0,policy_refund_001,manual_policy,Refund Policy,REFUND,refund_policy,When can a customer request a refund?,Customers may request a refund within 14 days ...,Refund Policy: Customers may request a refund ...
1,policy_delivery_001,manual_policy,Shipping and Delivery Policy,DELIVERY,shipping_policy,What should happen when an order has not arrived?,Standard delivery usually takes 3 to 7 busines...,Shipping and Delivery Policy: Standard deliver...
2,policy_damaged_001,manual_policy,Damaged Product Policy,PRODUCT,damaged_product_policy,What should happen if a product arrives damaged?,"If a customer receives a damaged product, they...",Damaged Product Policy: If a customer receives...
3,policy_billing_001,manual_policy,Billing and Payment Policy,BILLING,billing_policy,How should billing and duplicate charge issues...,"For duplicate charges, incorrect charges, or f...",Billing and Payment Policy: For duplicate char...
4,policy_account_001,manual_policy,Account and Login Support Policy,ACCOUNT,account_login_policy,What should happen if a customer cannot log in?,"If a customer cannot log in, they should first...",Account and Login Support Policy: If a custome...


## 5. Combine Dataset FAQ Entries and Manual Policies

This creates the final hybrid knowledge base for the RAG system.


In [7]:
knowledge_base = dataset_faq_entries + manual_policy_entries

kb_df = pd.DataFrame(knowledge_base)

print("Total knowledge base entries:", len(knowledge_base))
print("\nSource type counts:")
print(kb_df["source_type"].value_counts())

kb_df.head()


Total knowledge base entries: 56

Source type counts:
source_type
dataset_faq      48
manual_policy     8
Name: count, dtype: int64


,doc_id,source_type,title,category,intent,question,answer,text
0,dataset_faq_0000,dataset_faq,ORDER - cancel_order,ORDER,cancel_order,"I can't pay for this item, cancel order {{Orde...",I understand the urgency and frustration of no...,"Customer question: I can't pay for this item, ..."
1,dataset_faq_0001,dataset_faq,PAYMENT - check_payment_methods,PAYMENT,check_payment_methods,"I have got to list the payment modalities, hel...",Thank you for reaching out! I completely under...,Customer question: I have got to list the paym...
2,dataset_faq_0002,dataset_faq,SHIPPING - change_shipping_address,SHIPPING,change_shipping_address,can you tell me more about updating the address?,Of course! When it comes to updating your addr...,Customer question: can you tell me more about ...
3,dataset_faq_0003,dataset_faq,INVOICE - check_invoice,INVOICE,check_invoice,wanna see the bill from {{Person Name}},I'll get right on it! I understand your desire...,Customer question: wanna see the bill from {{P...
4,dataset_faq_0004,dataset_faq,INVOICE - check_invoice,INVOICE,check_invoice,ser the bill #85632,I'll do my best! I'm here to assist you with r...,Customer question: ser the bill #85632\nRecomm...


## 6. Prepare Searchable Documents and Metadata

FAISS stores vectors, but it does not store full document details by itself. This section prepares the searchable text for embedding and keeps metadata so retrieved results can still show the title, category, intent, answer, and source type.

So we create:

- `documents`: searchable text to embed
- `metadata`: details to show after retrieval


In [8]:
documents = []
metadata = []

for item in knowledge_base:
    searchable_text = (
        f"Title: {item['title']}. "
        f"Source type: {item['source_type']}. "
        f"Category: {item['category']}. "
        f"Intent: {item['intent']}. "
        f"Question: {item['question']}. "
        f"Answer: {item['answer']}."
    )

    documents.append(searchable_text)

    metadata.append({
        "doc_id": item["doc_id"],
        "source_type": item["source_type"],
        "title": item["title"],
        "category": item["category"],
        "intent": item["intent"],
        "question": item["question"],
        "answer": item["answer"],
        "text": item["text"]
    })

print("Total RAG documents:", len(documents))
print("\nExample searchable document:\n")
print(documents[0])


Total RAG documents: 56

Example searchable document:

Title: ORDER - cancel_order. Source type: dataset_faq. Category: ORDER. Intent: cancel_order. Question: I can't pay for this item, cancel order {{Order Number}}. Answer: I understand the urgency and frustration of not being able to pay for this item and the need to cancel order {{Order Number}}. We're here to assist you in resolving this issue swiftly. To cancel your order, please follow these steps:

1. Log into your {{Online Company Portal Info}} using your credentials.
2. Navigate to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
3. Locate the order with the number {{Order Number}} and click on it.
4. Look for the option labeled '{{Online Order Interaction}}' associated with your purchase and select it.
5. Follow any additional prompts or instructions to complete the cancellation process.

If you have any further questions or encounter any difficulties, please don't hesitate to reach out to our ded

## 7. Build Embeddings

The main embedding model is:

```text
sentence-transformers/all-MiniLM-L6-v2
```

This produces semantic embeddings for the FAQ and policy entries.

If SentenceTransformer is unavailable, the notebook uses a dense TF-IDF fallback. FAISS is still used for vector search.


In [9]:
def build_embeddings(texts):
    """
    Convert text documents into dense vectors.

    Primary:
        SentenceTransformer semantic embeddings.

    Fallback:
        TF-IDF dense vectors.
    """

    if SENTENCE_TRANSFORMERS_AVAILABLE:
        model = SentenceTransformer("all-MiniLM-L6-v2")
        embeddings = model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        embedding_info = {
            "method": "SentenceTransformer all-MiniLM-L6-v2",
            "model": model,
            "vectorizer": None
        }

    else:
        vectorizer = TfidfVectorizer(stop_words="english", max_features=2000)
        embeddings = vectorizer.fit_transform(texts).toarray().astype("float32")

        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        embeddings = embeddings / np.maximum(norms, 1e-12)

        embedding_info = {
            "method": "TF-IDF dense fallback",
            "model": None,
            "vectorizer": vectorizer
        }

    return embeddings.astype("float32"), embedding_info


doc_embeddings, embedding_info = build_embeddings(documents)

print("Embedding method:", embedding_info["method"])
print("Embedding shape:", doc_embeddings.shape)


Batches: 100%|██████████| 2/2 [00:00<00:00,  7.32it/s]

Embedding method: SentenceTransformer all-MiniLM-L6-v2
Embedding shape: (56, 384)


## 8. Build the FAISS Vector Database

This is the vector database for the RAG component.

The notebook uses `faiss.IndexFlatIP`.

Because embeddings are normalized, inner product similarity works like cosine similarity.


In [10]:
if not FAISS_AVAILABLE:
    raise ImportError("FAISS is required. Install it with: pip install faiss-cpu")

embedding_dim = doc_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(doc_embeddings)

print("FAISS index created successfully.")
print("Vector dimension:", embedding_dim)
print("Number of vectors stored:", faiss_index.ntotal)


FAISS index created successfully.
Vector dimension: 384
Number of vectors stored: 56


## 9. Query Embedding Function

This converts a new customer message into the same vector format as the knowledge base documents. Without this step, FAISS cannot compare the customer message with the stored FAQ and policy vectors.


In [11]:
def embed_query(query_text):
    """
    Convert a customer query into a normalized vector for FAISS search.
    """

    if embedding_info["method"].startswith("SentenceTransformer"):
        model = embedding_info["model"]
        query_embedding = model.encode(
            [query_text],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    else:
        vectorizer = embedding_info["vectorizer"]
        query_embedding = vectorizer.transform([query_text]).toarray().astype("float32")

        norms = np.linalg.norm(query_embedding, axis=1, keepdims=True)
        query_embedding = query_embedding / np.maximum(norms, 1e-12)

    return query_embedding.astype("float32")


## 10. FAISS RAG Retrieval Function

This is the main retrieval step of the RAG pipeline. It searches the hybrid knowledge base using the customer message and, when available, the Part 2 category or intent.

It uses:

- customer message
- optional Part 2 category
- optional Part 2 intent if available

Output is a list of relevant FAQ/policy entries.


In [12]:
def retrieve_context(customer_message, category=None, intent=None, top_k=5, min_score=0.25):
    """
    Retrieve relevant FAQ/policy entries using FAISS.

    Args:
        customer_message (str): Original customer message.
        category (str): Optional category from Part 2.
        intent (str): Optional intent from Part 2 if available.
        top_k (int): Number of results to retrieve.
        min_score (float): Minimum similarity score to keep.

    Returns:
        list[dict]: Retrieved documents with metadata and similarity scores.
    """

    query_parts = [customer_message]

    if category:
        query_parts.append(f"Category: {category}")

    if intent:
        query_parts.append(f"Intent: {intent}")

    query_text = " ".join(query_parts)

    query_embedding = embed_query(query_text)

    scores, indices = faiss_index.search(query_embedding, top_k)

    retrieved = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        item = metadata[int(idx)].copy()
        item["score"] = float(score)

        if item["score"] >= min_score:
            retrieved.append(item)

    return retrieved


# Quick retrieval test
test_message = "My order has not arrived and the tracking link is not working."
test_retrieval = retrieve_context(test_message, category="DELIVERY", top_k=5)

pd.DataFrame(test_retrieval)


,doc_id,source_type,title,category,intent,question,answer,text,score
0,policy_delivery_001,manual_policy,Shipping and Delivery Policy,DELIVERY,shipping_policy,What should happen when an order has not arrived?,Standard delivery usually takes 3 to 7 busines...,Shipping and Delivery Policy: Standard deliver...,0.462877
1,dataset_faq_0015,dataset_faq,SHIPPING - change_shipping_address,SHIPPING,change_shipping_address,I don't know what to do to correct my address,No worries! If you're unsure about how to corr...,Customer question: I don't know what to do to...,0.368795
2,dataset_faq_0032,dataset_faq,SHIPPING - change_shipping_address,SHIPPING,change_shipping_address,i have anissue modifying my address,I'm sorry to hear that you're experiencing dif...,Customer question: i have anissue modifying my...,0.350895
3,dataset_faq_0000,dataset_faq,ORDER - cancel_order,ORDER,cancel_order,"I can't pay for this item, cancel order {{Orde...",I understand the urgency and frustration of no...,"Customer question: I can't pay for this item, ...",0.322200
4,policy_damaged_001,manual_policy,Damaged Product Policy,PRODUCT,damaged_product_policy,What should happen if a product arrives damaged?,"If a customer receives a damaged product, they...",Damaged Product Policy: If a customer receives...,0.322104


## 11. Format Retrieved Context for Part 3 LLM Prompt

Person 3 can use this output in the final LLM prompt.

This is the main bridge between your Part 4 and Person 3's response generator.


In [13]:
def format_retrieved_context_for_prompt(retrieved_context):
    """
    Create a concise RAG context for P3.
    This gives the LLM guidance without letting it copy messy dataset text.
    """

    if not retrieved_context:
        return "No relevant company FAQ or policy was retrieved."

    clean_blocks = []

    for i, doc in enumerate(retrieved_context, start=1):
        title = doc.get("title", "Unknown")
        category = str(doc.get("category", "GENERAL")).upper()
        intent = str(doc.get("intent", "general")).lower()
        score = doc.get("score", 0)

        # Create clean guidance based on category/intent
        if "cancel" in intent or "cancel" in title.lower():
            guidance = (
                "For order cancellation requests, ask the customer for their order number. "
                "Explain that cancellation depends on whether the order has already been processed or shipped. "
                "If the order has already shipped, guide the customer to return or refund options."
            )

        elif "delivery" in category.lower() or "track" in intent or "shipping" in title.lower():
            guidance = (
                "For delivery issues, ask the customer for their order number and delivery address. "
                "Explain that the support team can check the tracking status and investigate delayed or missing packages."
            )

        elif "refund" in category.lower() or "refund" in intent:
            guidance = (
                "For refund requests, ask for the order number and the reason for the refund. "
                "If the product is damaged, ask for a photo. Do not guarantee a refund before review."
            )

        elif "billing" in category.lower() or "payment" in intent or "invoice" in intent:
            guidance = (
                "For billing or payment issues, ask for the order number, transaction date, and payment receipt or screenshot. "
                "Explain that the billing team will review the charge."
            )

        elif "account" in category.lower() or "login" in intent:
            guidance = (
                "For account issues, ask the customer to confirm their registered email address. "
                "Suggest password reset or account verification where appropriate."
            )

        else:
            answer = doc.get("answer", "")
            answer = answer.replace("[details to be confirmed]", "").strip()
            sentences = re.split(r"(?<=[.!?])\s+", answer)
            guidance = " ".join(sentences[:2]).strip()

            if not guidance:
                guidance = "Use the retrieved document as general support guidance."

        clean_blocks.append(
            f"[Context {i}: {title} | Category: {category} | Intent: {intent} | Score: {score:.3f}]\n"
            f"Guidance: {guidance}"
        )

    return "\n\n".join(clean_blocks)

## 12. Escalation Classifier

Not every customer issue should be handled automatically. This section checks for serious cases, such as negative sentiment, billing issues, delivery complaints, repeated unresolved problems, legal keywords, or manager requests.

It uses:

- sentiment from Part 2
- category from Part 2
- serious words in the customer message
- whether RAG found useful context
- retrieval confidence


In [14]:
def check_escalation(customer_message, p2_analysis=None, retrieved_context=None):
    """
    Decide whether the customer case should be escalated to a human agent.
    """

    if p2_analysis is None:
        p2_analysis = {}

    if retrieved_context is None:
        retrieved_context = []

    text = customer_message.lower()

    escalation_keywords = [
        "legal", "lawyer", "lawsuit", "court", "ombudsman",
        "manager", "supervisor", "human agent", "real person",
        "angry", "furious", "unacceptable", "terrible service",
        "scam", "fraud", "stolen", "charged twice",
        "not resolved", "again", "third time", "fourth time",
        "formal complaint", "cancel my account"
    ]

    keyword_matches = [word for word in escalation_keywords if word in text]

    sentiment = str(p2_analysis.get("sentiment", "")).lower()
    category = str(p2_analysis.get("category", "")).lower()

    negative_sentiment = sentiment in [
        "negative", "angry", "very negative", "frustrated", "complaint"
    ]

    serious_categories = [
        "billing", "refund", "delivery", "technical", "account",
        "complaint", "invoice", "payment", "order"
    ]

    serious_category = any(cat in category for cat in serious_categories)

    no_retrieval = len(retrieved_context) == 0

    low_retrieval_confidence = False
    if retrieved_context:
        best_score = max(doc.get("score", 0) for doc in retrieved_context)
        low_retrieval_confidence = best_score < 0.25

    reasons = []

    if keyword_matches:
        reasons.append(f"Escalation keywords detected: {', '.join(keyword_matches)}")

    if negative_sentiment and serious_category:
        reasons.append("Negative sentiment detected for a serious customer service category.")

    if no_retrieval:
        reasons.append("No relevant FAQ or policy was retrieved.")

    if low_retrieval_confidence:
        reasons.append("The best retrieved result has low confidence.")

    escalate = len(reasons) > 0

    if not reasons:
        reasons.append("The case appears suitable for automated support.")

    return {
        "escalate": escalate,
        "reasons": reasons,
        "sentiment": sentiment,
        "category": category
    }


sample_p2_analysis = {
    "sentiment": "negative",
    "category": "DELIVERY",
    "entities": {"ORDER_ID": [], "PRODUCT": []}
}

check_escalation(test_message, sample_p2_analysis, test_retrieval)


{'escalate': True,
 'reasons': ['Negative sentiment detected for a serious customer service category.'],
 'sentiment': 'negative',
 'category': 'delivery'}

## 13. Agent Decision Logic

This section decides what the chatbot should do next. The agent can use retrieved information to respond, ask for more details, or escalate the case to a human agent.

Possible actions:

- `RETRIEVE_THEN_RESPOND`
- `ESCALATE_TO_HUMAN`
- `ASK_FOR_MORE_INFORMATION`
- `DIRECT_RESPONSE`


In [15]:
def agent_decision(customer_message, p2_analysis=None, retrieved_context=None):
    """
    Decide the next action for the assistant.
    """

    if p2_analysis is None:
        p2_analysis = {}

    if retrieved_context is None:
        retrieved_context = []

    escalation_output = check_escalation(customer_message, p2_analysis, retrieved_context)

    if escalation_output["escalate"]:
        return {
            "action": "ESCALATE_TO_HUMAN",
            "reason": "Human review is recommended. " + " ".join(escalation_output["reasons"]),
            "escalation": escalation_output
        }

    if len(retrieved_context) == 0:
        return {
            "action": "ASK_FOR_MORE_INFORMATION",
            "reason": "No relevant FAQ or policy was retrieved, so the assistant should ask for more details.",
            "escalation": escalation_output
        }

    best_score = max(doc.get("score", 0) for doc in retrieved_context)

    if best_score >= 0.25:
        return {
            "action": "RETRIEVE_THEN_RESPOND",
            "reason": "Relevant FAQ or policy information was retrieved and can be used for response generation.",
            "escalation": escalation_output
        }

    return {
        "action": "DIRECT_RESPONSE",
        "reason": "The issue seems simple, but retrieval confidence is not high enough for strong policy grounding.",
        "escalation": escalation_output
    }


agent_decision(test_message, sample_p2_analysis, test_retrieval)


{'action': 'ESCALATE_TO_HUMAN',
 'reason': 'Human review is recommended. Negative sentiment detected for a serious customer service category.',
 'escalation': {'escalate': True,
  'reasons': ['Negative sentiment detected for a serious customer service category.'],
  'sentiment': 'negative',
  'category': 'delivery'}}

## 14. Dialogue Tracking

 This section records each customer interaction, including the message, Part 2 analysis, retrieved documents, agent decision, and escalation result. This supports conversation memory and can later be connected to Streamlit or SQLite.

In the final Streamlit/Gradio system, Person 5 can move this into:

- Streamlit session state
- SQLite database
- JSON file


In [16]:
dialogue_history = []

def update_dialogue_history(customer_message, p2_analysis, retrieved_context, agent_output, bot_response=None):
    """
    Store one customer interaction in the dialogue history.
    """

    record = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "customer_message": customer_message,
        "p2_analysis": p2_analysis,
        "retrieved_titles": [doc["title"] for doc in retrieved_context],
        "retrieved_source_types": [doc["source_type"] for doc in retrieved_context],
        "agent_action": agent_output.get("action"),
        "agent_reason": agent_output.get("reason"),
        "escalate": agent_output.get("escalation", {}).get("escalate"),
        "bot_response": bot_response
    }

    dialogue_history.append(record)

    return dialogue_history


## 15. Final Part 4 Integration Function

This section provides one clean function that other team members can use. Person 5 can call this function to connect Part 2 analysis, Part 4 retrieval/agent logic, and Part 3 response generation.

Expected final system:

```python
p2_analysis = analyze(customer_message)

part4_output = run_part4_rag_agent(customer_message, p2_analysis)

final_reply = generate_customer_reply(
    customer_message,
    p2_analysis,
    part4_output["retrieved_context_text"],
    part4_output["agent_output"]
)
```


In [17]:
def run_part4_rag_agent(customer_message, p2_analysis=None, top_k=5):
    """
    Main Part 4 function for integration.

    Args:
        customer_message (str): Original customer query.
        p2_analysis (dict): Output from Part 2 analyze() function.
        top_k (int): Number of RAG documents to retrieve.

    Returns:
        dict: Retrieved context, formatted context, escalation output, agent output, and dialogue history.
    """

    if p2_analysis is None:
        p2_analysis = {}

    category = p2_analysis.get("category")
    intent = p2_analysis.get("intent")

    retrieved_context = retrieve_context(
        customer_message=customer_message,
        category=category,
        intent=intent,
        top_k=top_k
    )

    escalation_output = check_escalation(
        customer_message=customer_message,
        p2_analysis=p2_analysis,
        retrieved_context=retrieved_context
    )

    agent_output = agent_decision(
        customer_message=customer_message,
        p2_analysis=p2_analysis,
        retrieved_context=retrieved_context
    )

    retrieved_context_text = format_retrieved_context_for_prompt(retrieved_context)

    update_dialogue_history(
        customer_message=customer_message,
        p2_analysis=p2_analysis,
        retrieved_context=retrieved_context,
        agent_output=agent_output,
        bot_response=None
    )

    return {
        "retrieved_context": retrieved_context,
        "retrieved_context_text": retrieved_context_text,
        "escalation_output": escalation_output,
        "agent_output": agent_output,
        "dialogue_history": dialogue_history
    }


# Example run
example_output = run_part4_rag_agent(
    customer_message="My order has not arrived and the tracking link is not working.",
    p2_analysis={
        "sentiment": "negative",
        "category": "DELIVERY",
        "entities": {"ORDER_ID": [], "PRODUCT": []}
    }
)

example_output


{'retrieved_context': [{'doc_id': 'policy_delivery_001',
   'source_type': 'manual_policy',
   'title': 'Shipping and Delivery Policy',
   'category': 'DELIVERY',
   'intent': 'shipping_policy',
   'question': 'What should happen when an order has not arrived?',
   'answer': 'Standard delivery usually takes 3 to 7 business days. If an order has not arrived after the estimated delivery date, customers should provide the order ID and delivery address. The case may be escalated to the logistics team if tracking is unavailable or the package appears lost.',
   'text': 'Shipping and Delivery Policy: Standard delivery usually takes 3 to 7 business days. If an order has not arrived after the estimated delivery date, customers should provide the order ID and delivery address. The case may be escalated to the logistics team if tracking is unavailable or the package appears lost.',
   'score': 0.4628767967224121},
  {'doc_id': 'dataset_faq_0015',
   'source_type': 'dataset_faq',
   'title': 'SHI

## 16. Test Cases and Evaluation Table

This table demonstrates whether the RAG + agent module works across different customer service situations.


In [18]:
test_cases = [
    {
        "customer_message": "My order has not arrived even though the delivery date passed yesterday.",
        "p2_analysis": {
            "sentiment": "negative",
            "category": "DELIVERY",
            "entities": {"ORDER_ID": [], "PRODUCT": []}
        }
    },
    {
        "customer_message": "I received a damaged laptop and I want a refund.",
        "p2_analysis": {
            "sentiment": "negative",
            "category": "REFUND",
            "entities": {"PRODUCT": ["laptop"]}
        }
    },
    {
        "customer_message": "I was charged twice for the same order. This is unacceptable.",
        "p2_analysis": {
            "sentiment": "negative",
            "category": "BILLING",
            "entities": {"MONEY": [], "ORDER_ID": []}
        }
    },
    {
        "customer_message": "I cannot log into my account after resetting my password.",
        "p2_analysis": {
            "sentiment": "neutral",
            "category": "ACCOUNT",
            "entities": {}
        }
    },
    {
        "customer_message": "I need an invoice for my last order.",
        "p2_analysis": {
            "sentiment": "neutral",
            "category": "INVOICE",
            "entities": {}
        }
    },
    {
        "customer_message": "I contacted support three times and nobody fixed my issue. I want a manager.",
        "p2_analysis": {
            "sentiment": "angry",
            "category": "COMPLAINT",
            "entities": {}
        }
    }
]

evaluation_results = []

for case in test_cases:
    output = run_part4_rag_agent(
        customer_message=case["customer_message"],
        p2_analysis=case["p2_analysis"]
    )

    retrieved_titles = [doc["title"] for doc in output["retrieved_context"]]
    source_types = [doc["source_type"] for doc in output["retrieved_context"]]
    top_score = output["retrieved_context"][0]["score"] if output["retrieved_context"] else None

    evaluation_results.append({
        "customer_message": case["customer_message"],
        "category": case["p2_analysis"].get("category"),
        "sentiment": case["p2_analysis"].get("sentiment"),
        "top_retrieved_title": retrieved_titles[0] if retrieved_titles else "None",
        "retrieved_source_types": ", ".join(source_types[:3]),
        "top_score": top_score,
        "agent_action": output["agent_output"]["action"],
        "escalate": output["escalation_output"]["escalate"],
        "reason": output["agent_output"]["reason"]
    })

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df


,customer_message,category,sentiment,top_retrieved_title,retrieved_source_types,top_score,agent_action,escalate,reason
0,My order has not arrived even though the deliv...,DELIVERY,negative,Shipping and Delivery Policy,"manual_policy, dataset_faq, manual_policy",0.564167,ESCALATE_TO_HUMAN,True,Human review is recommended. Negative sentimen...
1,I received a damaged laptop and I want a refund.,REFUND,negative,Refund Policy,"manual_policy, manual_policy, dataset_faq",0.453196,ESCALATE_TO_HUMAN,True,Human review is recommended. Negative sentimen...
2,I was charged twice for the same order. This i...,BILLING,negative,Billing and Payment Policy,"manual_policy, dataset_faq, dataset_faq",0.669046,ESCALATE_TO_HUMAN,True,Human review is recommended. Escalation keywor...
3,I cannot log into my account after resetting m...,ACCOUNT,neutral,Account and Login Support Policy,manual_policy,0.466788,RETRIEVE_THEN_RESPOND,False,Relevant FAQ or policy information was retriev...
4,I need an invoice for my last order.,INVOICE,neutral,INVOICE - check_invoice,"dataset_faq, dataset_faq, dataset_faq",0.573730,RETRIEVE_THEN_RESPOND,False,Relevant FAQ or policy information was retriev...
5,I contacted support three times and nobody fix...,COMPLAINT,angry,Technical Support Policy,"manual_policy, dataset_faq, dataset_faq",0.340626,ESCALATE_TO_HUMAN,True,Human review is recommended. Escalation keywor...


## 17. P4 Quantitative Evaluation

This section evaluates the RAG and agent component using measurable results. The evaluation checks whether the retrieved context matches the expected customer service policy and whether the escalation decision is correct for each test case.

The main metrics used are:

- **Retrieval accuracy:** how often the correct policy/context was retrieved.
- **Escalation accuracy:** how often the agent made the correct escalation decision.
- **Average response time:** how long Part 4 takes to retrieve context and make an agent decision.

In [19]:
import time

evaluation_cases = [
    {
        "message": "I want to cancel my order.",
        "p2": {"category": "ORDER", "sentiment": "neutral", "entities": {}},
        "expected_policy_keyword": "cancel",
        "expected_escalation": False
    },
    {
        "message": "My MacBook arrived damaged.",
        "p2": {"category": "PRODUCT", "sentiment": "negative", "entities": {}},
        "expected_policy_keyword": "damaged",
        "expected_escalation": False
    },
    {
        "message": "I was charged twice for the same order.",
        "p2": {"category": "BILLING", "sentiment": "negative", "entities": {}},
        "expected_policy_keyword": "billing",
        "expected_escalation": True
    },
    {
        "message": "My package has not arrived yet.",
        "p2": {"category": "DELIVERY", "sentiment": "negative", "entities": {}},
        "expected_policy_keyword": "delivery",
        "expected_escalation": False
    },
    {
        "message": "I have contacted support three times and nobody helped me. I want to speak to a manager.",
        "p2": {"category": "COMPLAINT", "sentiment": "negative", "entities": {}},
        "expected_policy_keyword": "complaint",
        "expected_escalation": True
    }
]

evaluation_results = []

for case in evaluation_cases:
    start_time = time.time()

    output = run_part4_rag_agent(
        case["message"],
        case["p2"],
        top_k=2
    )

    response_time = time.time() - start_time

    retrieved_text = output.get("retrieved_context_text", "").lower()
    actual_escalation = output.get("escalation_output", {}).get("escalate", False)

    retrieval_correct = case["expected_policy_keyword"].lower() in retrieved_text
    escalation_correct = actual_escalation == case["expected_escalation"]

    evaluation_results.append({
        "Customer message": case["message"],
        "Expected policy keyword": case["expected_policy_keyword"],
        "Retrieval correct": retrieval_correct,
        "Expected escalation": case["expected_escalation"],
        "Actual escalation": actual_escalation,
        "Escalation correct": escalation_correct,
        "Agent action": output.get("agent_output", {}).get("action", "UNKNOWN"),
        "Response time (sec)": round(response_time, 4)
    })

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df

,Customer message,Expected policy keyword,Retrieval correct,Expected escalation,Actual escalation,Escalation correct,Agent action,Response time (sec)
0,I want to cancel my order.,cancel,True,False,False,True,RETRIEVE_THEN_RESPOND,0.0192
1,My MacBook arrived damaged.,damaged,True,False,False,True,RETRIEVE_THEN_RESPOND,0.0060
2,I was charged twice for the same order.,billing,True,True,True,True,ESCALATE_TO_HUMAN,0.0074
3,My package has not arrived yet.,delivery,True,False,True,False,ESCALATE_TO_HUMAN,0.0061
4,I have contacted support three times and nobod...,complaint,False,True,True,True,ESCALATE_TO_HUMAN,0.0068


In [20]:
retrieval_accuracy = evaluation_df["Retrieval correct"].mean() * 100
escalation_accuracy = evaluation_df["Escalation correct"].mean() * 100
average_response_time = evaluation_df["Response time (sec)"].mean()

print(f"Retrieval accuracy: {retrieval_accuracy:.2f}%")
print(f"Escalation accuracy: {escalation_accuracy:.2f}%")
print(f"Average Part 4 response time: {average_response_time:.4f} seconds")

Retrieval accuracy: 80.00%
Escalation accuracy: 80.00%
Average Part 4 response time: 0.0091 seconds


### P4 Evaluation Analysis

The Part 4 RAG and agent module was evaluated using five customer service test cases covering order cancellation, damaged products, billing issues, delivery issues, and complaint escalation. The retrieval accuracy was **80%**, meaning the system retrieved the expected policy context for four out of five test cases. This shows that the FAISS-based retrieval system is generally able to find relevant support guidance for common customer issues.

The escalation accuracy was also **80%**. The agent correctly handled simple order cancellation and damaged product cases without escalation, and correctly escalated the billing and complaint-related cases. However, the delivery delay case was incorrectly escalated even though the expected decision was not to escalate. This suggests that the escalation rules may be slightly too sensitive when the sentiment is negative, especially for common delivery problems that can usually be handled automatically.

The average Part 4 response time was **0.0091 seconds**, which shows that the RAG retrieval and agent decision process is very fast.. Overall, the Part 4 component performs efficiently and gives mostly correct retrieval and escalation decisions, but the escalation logic could be improved by making the rules more specific for delivery issues and repeated complaints.

## 18. Show Dialogue History

This confirms that dialogue tracking is working.


In [21]:
dialogue_df = pd.DataFrame(dialogue_history)
dialogue_df.tail()


,timestamp,customer_message,p2_analysis,retrieved_titles,retrieved_source_types,agent_action,agent_reason,escalate,bot_response
7,2026-06-03T20:53:32,I want to cancel my order.,"{'category': 'ORDER', 'sentiment': 'neutral', ...","[ORDER - cancel_order, ORDER - cancel_order]","[dataset_faq, dataset_faq]",RETRIEVE_THEN_RESPOND,Relevant FAQ or policy information was retriev...,False,None
8,2026-06-03T20:53:32,My MacBook arrived damaged.,"{'category': 'PRODUCT', 'sentiment': 'negative...",[Damaged Product Policy],[manual_policy],RETRIEVE_THEN_RESPOND,Relevant FAQ or policy information was retriev...,False,None
9,2026-06-03T20:53:32,I was charged twice for the same order.,"{'category': 'BILLING', 'sentiment': 'negative...","[Billing and Payment Policy, INVOICE - check_i...","[manual_policy, dataset_faq]",ESCALATE_TO_HUMAN,Human review is recommended. Escalation keywor...,True,None
10,2026-06-03T20:53:32,My package has not arrived yet.,"{'category': 'DELIVERY', 'sentiment': 'negativ...","[Shipping and Delivery Policy, Damaged Product...","[manual_policy, manual_policy]",ESCALATE_TO_HUMAN,Human review is recommended. Negative sentimen...,True,None
11,2026-06-03T20:53:32,I have contacted support three times and nobod...,"{'category': 'COMPLAINT', 'sentiment': 'negati...","[Technical Support Policy, ORDER - cancel_order]","[manual_policy, dataset_faq]",ESCALATE_TO_HUMAN,Human review is recommended. Escalation keywor...,True,None


## 19. Integration Package for Part 3

The most useful value for Person 3 is:

```python
part4_output["retrieved_context_text"]
```

This text can be placed inside the LLM prompt so the response is grounded in retrieved FAQ/policy information.


In [22]:
part3_input_package = {
    "customer_message": "I received a damaged phone and I want a refund.",
    "p2_analysis": {
        "sentiment": "negative",
        "category": "REFUND",
        "entities": {"PRODUCT": ["phone"]}
    }
}

part4_for_part3 = run_part4_rag_agent(
    part3_input_package["customer_message"],
    part3_input_package["p2_analysis"]
)

print("Agent action:", part4_for_part3["agent_output"]["action"])
print("Escalate:", part4_for_part3["escalation_output"]["escalate"])
print("\nRetrieved context for Part 3:\n")
print(part4_for_part3["retrieved_context_text"])


Agent action: ESCALATE_TO_HUMAN
Escalate: True

Retrieved context for Part 3:

[Context 1: Refund Policy | Category: REFUND | Intent: refund_policy | Score: 0.430]
Guidance: For refund requests, ask for the order number and the reason for the refund. If the product is damaged, ask for a photo. Do not guarantee a refund before review.

[Context 2: ORDER - cancel_order | Category: ORDER | Intent: cancel_order | Score: 0.345]
Guidance: For order cancellation requests, ask the customer for their order number. Explain that cancellation depends on whether the order has already been processed or shipped. If the order has already shipped, guide the customer to return or refund options.

[Context 3: Damaged Product Policy | Category: PRODUCT | Intent: damaged_product_policy | Score: 0.342]
Guidance: If a customer receives a damaged product, they should contact support within 7 days of delivery. They may be asked to provide photos of the damaged item, the order ID, and a short description of the

## 20. Report-Ready Explanation

### RAG Knowledge Base Construction

The RAG knowledge base was built using a hybrid design. First, rows from the Bitext customer support dataset were converted into FAQ-style entries. Each entry contains a customer instruction, support response, category, and intent. This makes the knowledge base realistic because it uses customer-service-style examples similar to the input messages handled by the system.

Second, manually written company policy entries were added to cover official support rules such as refund policy, delivery policy, billing policy, account support, technical troubleshooting, privacy, and escalation. This hybrid design was chosen because dataset examples improve semantic matching, while manual policy entries provide clearer company rules for the generated response.

### FAISS Vector Database

Each FAQ and policy entry was converted into a vector embedding using a sentence embedding model. The embeddings were stored in a FAISS vector index using `IndexFlatIP`. Because the embeddings were normalized, inner product similarity was used like cosine similarity. When a new customer inquiry is received, the inquiry is embedded and searched against the FAISS index to retrieve the most relevant FAQ or policy entries.

### RAG Retrieval Pipeline

The retrieval function uses the original customer message and, when available, the category predicted by Part 2. The top retrieved FAQ/policy entries are returned with their metadata and similarity scores. These retrieved results are then formatted into a prompt-ready context block for the LLM response generation component in Part 3.

### Agentic Decision Logic

The agent decision module chooses whether the assistant should respond using retrieved information, ask for more details, or escalate the case to a human agent. This creates a simple agentic workflow where the system takes different actions depending on the customer message, sentiment, category, and retrieval result.

### Escalation Classifier

The escalation classifier uses explainable rules. It escalates cases when the customer message contains serious keywords such as legal action, manager request, repeated unresolved issue, duplicate charge, or fraud. It also considers negative sentiment in serious categories such as billing, refund, delivery, technical support, account issues, complaints, and payment problems.

### Dialogue Tracking

The dialogue history stores each interaction, including the customer message, Part 2 analysis, retrieved documents, source types, agent decision, escalation decision, and final response placeholder. This can later be connected to Streamlit session state or SQLite by the system integrator.

### Limitations

Although the knowledge base is more structured than a simple manual policy list, its effectiveness still depends on the quality of the dataset and the examples selected for retrieval. Some responses from the dataset may be quite general, and the manually written policies are simplified compared to full real-world company support documents. In future work, the system could be improved by using a larger and more carefully curated FAQ collection, real company support documents, better document chunking, reranking methods, and a trained escalation classifier.

## 21. Self-Evaluation for Part 4

For my contribution, I implemented the RAG and agent decision-making component of the intelligent customer service system. My main responsibility was to build the part of the system that retrieves relevant support information and decides how the chatbot should handle each customer inquiry. To do this, I developed a hybrid knowledge base by combining dataset-based FAQ entries with manually written company policy documents. The dataset rows were converted into customer-support FAQ entries containing the customer instruction, support response, category, and intent. This helped the knowledge base reflect realistic customer service examples while still keeping the structure useful for retrieval.

After creating the knowledge base, I generated embeddings for each FAQ and policy entry and stored them in a FAISS vector database. This allowed the system to perform semantic retrieval, meaning it can find relevant information based on the meaning of the customer message rather than only exact keyword matching. When a customer submits a question, the retrieval function searches the FAISS index and returns the most relevant FAQ or policy entries. These retrieved results are then formatted into a clear context block that can be passed to the LLM in Part 3 for response generation.

I also implemented the agent decision logic and escalation classifier. The escalation classifier checks whether a case should be handled by a human agent based on factors such as negative sentiment, serious categories like billing or delivery, repeated unresolved issues, manager requests, or legal-related keywords. The agent decision function then decides whether the chatbot should generate a response using retrieved information, ask for more details, or escalate the case to a human support agent. This makes the chatbot more practical because it does not treat every customer message the same way.

In addition, I added dialogue tracking, test cases, and a final integration function so that my part can connect smoothly with the rest of the project. The final function takes the original customer message and the Part 2 NLP analysis as input, then returns the retrieved context, escalation result, and agent action. Overall, my part connects the Part 2 NLP analysis with the Part 3 LLM response generation, allowing the chatbot to produce more grounded, policy-aware, and contextually appropriate customer service responses instead of relying only on generic LLM output.
